In [1]:
!pip install faster-whisper gradio transformers accelerate \
    torch torchaudio groq gTTS pydub soundfile numpy

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached gradio-6.15.0-py3-none-any.whl.metadata (17 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached fastapi-0.136.3-py3-none-any.whl.metadata (27 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
INFO: pip is looking at multiple versions of gtts to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gtts to determine which version is compatible with other requirements. This could take a while.
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
Using cached gradio-6.15.0-py3-none-any.whl (20.1 MB)
Using cached gradio_client-2.5.0-py3-none-any.whl (59 kB)
Using cached fastapi-0.136.3-py3-none-any.whl (117 kB)
Using cached hf

In [5]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Wed May 27 05:35:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A30                     On  |   00000000:01:00.0 Off |                    0 |
| N/A   27C    P0             26W /  165W |       0MiB /  24576MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
from faster_whisper import WhisperModel

# Load model (medium is fast + accurate on A30)
asr_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("✅ ASR Model Loaded!")

✅ ASR Model Loaded!


In [7]:
import os

# Set it as environment variable - safer
os.environ["GROQ_API_KEY"] = "Your_personal_GROQ_KEY"

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def get_llm_response(text):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are a helpful study tutor assistant. 
            Help students understand concepts clearly and concisely.
            Keep responses under 3 sentences for voice output."""},
            {"role": "user", "content": text}
        ]
    )
    return response.choices[0].message.content

print(get_llm_response("Explain Newton's first law simply"))

Newton's first law, also known as the law of inertia, states that an object at rest will remain at rest, and an object in motion will continue moving with the same speed and direction unless acted upon by an external force.


In [8]:
from gtts import gTTS
import IPython.display as ipd

def text_to_speech(text, filename="response.mp3"):
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save(filename)
    return filename

# Test it
text_to_speech("Hello! I am your study tutor. How can I help you today?")
ipd.Audio("response.mp3")

In [9]:
import time
import tempfile
import os

def run_pipeline(audio_file_path):
    
    # Stage 1: ASR
    t1 = time.time()
    segments, _ = asr_model.transcribe(audio_file_path, beam_size=5)
    transcript = " ".join([s.text for s in segments])
    t2 = time.time()
    print(f"📝 Transcript: {transcript}")
    print(f"⏱️ ASR Latency: {t2-t1:.2f}s")
    
    # Stage 2: LLM
    response_text = get_llm_response(transcript)
    t3 = time.time()
    print(f"🤖 Response: {response_text}")
    print(f"⏱️ LLM Latency: {t3-t2:.2f}s")
    
    # Stage 3: TTS
    output_file = text_to_speech(response_text)
    t4 = time.time()
    print(f"⏱️ TTS Latency: {t4-t3:.2f}s")
    print(f"⏱️ Total Latency: {t4-t1:.2f}s")
    
    return output_file, transcript, response_text

print("✅ Pipeline ready!")

✅ Pipeline ready!


In [14]:
# Run this in a new cell to create a test audio file
from gtts import gTTS

test_text = "Explain Newton's first law of motion"
tts = gTTS(text=test_text, lang='en')
tts.save("/tmp/test_question.mp3")
print("✅ Test audio created at /tmp/test_question.mp3")

Task was destroyed but it is pending!
task: <Task pending name='Task-250' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-251' coro=<_async_in_context.<locals>.preserve_context() running at /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/root/miniconda3/envs/py3.10/lib/python3.10/abc.py:123: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  return _abc_subclasscheck(cls, subclass)
Task was destroyed but it is pending!
task: <Task pending name='Task-251' coro=<_async_in_context.<locals>.preserve_context() running at /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.tas

✅ Test audio created at /tmp/test_question.mp3


In [15]:
# This will show a download link
from IPython.display import FileLink
FileLink('/tmp/test_question.mp3')

/tmp/test_question.mp3

/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/pydub/utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/gradio/processing_utils.py", line 657, in audio_from_file
    audio = AudioSegment.from_file(filename)
  File "/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/pydub/audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/pydub/utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "/root/miniconda3/envs/py3.10/lib/python3.10/subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,


In [2]:
import os

# Fix broken dpkg first
os.system("dpkg --configure -a")
os.system("apt-get install -f -y")

# Now install ffmpeg
result = os.system("apt-get install -y ffmpeg")
print(f"ffmpeg install result: {result}")

# Verify
result2 = os.system("ffmpeg -version")
print("✅ Done!" if result2 == 0 else "❌ Still failing")

Setting up xkb-data (2.33-1) ...
Setting up libgomp1:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up libasound2-data (1.2.6.1-1ubuntu1.1) ...
Setting up libslang2:amd64 (2.3.2-5build4) ...
Setting up libfribidi0:amd64 (1.0.8-2ubuntu3.1) ...
Setting up libquadmath0:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up shared-mime-info (2.1-2) ...
Setting up libpng16-16:amd64 (1.6.37-3ubuntu0.5) ...
Setting up libatomic1:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up libubsan1:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up libnuma1:amd64 (2.0.14-3ubuntu2) ...
Setting up alsa-topology-conf (1.2.5.1-2) ...
Setting up libasound2:amd64 (1.2.6.1-1ubuntu1.1) ...
Setting up libusb-1.0-0:amd64 (2:1.0.25-1ubuntu2) ...
Setting up libcc1-0:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up liblsan0:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up libitm1:amd64 (12.3.0-1ubuntu1~22.04.3) ...
Setting up alsa-ucm-conf (1.2.6.3-1ubuntu1.12) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
Reading package li

debconf: delaying package configuration, since apt-utils is not installed


Fetched 101 MB in 3s (38.8 MB/s)
Selecting previously unselected package libaom3:amd64.
(Reading database ... 31218 files and directories currently installed.)
Preparing to unpack .../000-libaom3_3.3.0-1ubuntu0.1_amd64.deb ...
Unpacking libaom3:amd64 (3.3.0-1ubuntu0.1) ...
Selecting previously unselected package libva2:amd64.
Preparing to unpack .../001-libva2_2.14.0-1_amd64.deb ...
Unpacking libva2:amd64 (2.14.0-1) ...
Selecting previously unselected package libmfx1:amd64.
Preparing to unpack .../002-libmfx1_22.3.0-1_amd64.deb ...
Unpacking libmfx1:amd64 (22.3.0-1) ...
Selecting previously unselected package libva-drm2:amd64.
Preparing to unpack .../003-libva-drm2_2.14.0-1_amd64.deb ...
Unpacking libva-drm2:amd64 (2.14.0-1) ...
Selecting previously unselected package libva-x11-2:amd64.
Preparing to unpack .../004-libva-x11-2_2.14.0-1_amd64.deb ...
Unpacking libva-x11-2:amd64 (2.14.0-1) ...
Selecting previously unselected package libvdpau1:amd64.
Preparing to unpack .../005-libvdpau1_1

In [3]:
import gradio as gr
import numpy as np
import soundfile as sf
import time
import os

def text_to_speech(text):
    mp3_path = "/tmp/response.mp3"
    wav_path = "/tmp/response.wav"
    from gtts import gTTS
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save(mp3_path)
    os.system(f"ffmpeg -y -i {mp3_path} {wav_path}")
    return wav_path

def voice_assistant(audio):
    try:
        if audio is None:
            return None, "No audio received", "Please record something"
        
        sample_rate, audio_data = audio
        
        # Convert stereo to mono
        if len(audio_data.shape) > 1:
            audio_data = audio_data.mean(axis=1)
        
        # Normalize
        audio_data = audio_data.astype(np.float32)
        if np.abs(audio_data).max() > 1.0:
            audio_data = audio_data / 32768.0
        
        # Save as WAV
        tmp_wav = "/tmp/input_audio.wav"
        sf.write(tmp_wav, audio_data, sample_rate)
        print(f"✅ Audio saved: {sample_rate}Hz")
        
        # Stage 1: ASR
        t1 = time.time()
        segments, _ = asr_model.transcribe(tmp_wav, beam_size=5)
        transcript = " ".join([s.text for s in segments])
        t2 = time.time()
        print(f"📝 ASR ({t2-t1:.2f}s): {transcript}")
        
        if not transcript.strip():
            return None, "Could not understand audio", "Please speak clearly"
        
        # Stage 2: LLM
        response_text = get_llm_response(transcript)
        t3 = time.time()
        print(f"🤖 LLM ({t3-t2:.2f}s): {response_text}")
        
        # Stage 3: TTS
        output_file = text_to_speech(response_text)
        t4 = time.time()
        print(f"🔊 TTS ({t4-t3:.2f}s)")
        print(f"⏱️ TOTAL LATENCY: {t4-t1:.2f}s")
        
        return output_file, f"You said: {transcript}", f"Tutor: {response_text}"
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"Error: {str(e)}", "Please try again"

# Close any existing app
try:
    app.close()
except:
    pass

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Speak your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Speak your question and get an instant voice response from your AI tutor!"
)

app.launch(share=True)

* Running on local URL:  http://0.0.0.0:6006
* Running on public URL: https://68ba57b5ce9df43781.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
def text_to_speech(text):
    mp3_path = "/tmp/response.mp3"
    wav_path = "/tmp/response.wav"
    gTTS(text=text, lang='en').save(mp3_path)
    os.system(f"ffmpeg -y -i {mp3_path} {wav_path}")
    return wav_path

# Test TTS
out = text_to_speech("Hello I am your study tutor!")
import IPython.display as ipd
ipd.Audio(out)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [15]:
def text_to_speech(text):
    mp3_path = "/tmp/response.mp3"
    wav_path = "/tmp/response.wav"
    gTTS(text=text, lang='en').save(mp3_path)
    os.system(f"ffmpeg -y -i {mp3_path} -ar 22050 -ac 1 -sample_fmt s16 {wav_path}")
    return wav_path

In [16]:
# Create test audio
gTTS("Explain Newton's first law of motion", lang='en').save("/tmp/test_input.mp3")
os.system("ffmpeg -y -i /tmp/test_input.mp3 -ar 16000 /tmp/test_input.wav")

# Load it
audio_data, sr = sf.read("/tmp/test_input.wav")
print(f"✅ Audio: {len(audio_data)} samples at {sr}Hz")

# ASR
t1 = time.time()
segments, _ = asr_model.transcribe("/tmp/test_input.wav", beam_size=5)
transcript = " ".join([s.text for s in segments])
t2 = time.time()
print(f"📝 Transcript ({t2-t1:.2f}s): {transcript}")

# LLM
response_text = get_llm_response(transcript)
t3 = time.time()
print(f"🤖 Response ({t3-t2:.2f}s): {response_text}")

# TTS
out = text_to_speech(response_text)
t4 = time.time()
print(f"🔊 TTS ({t4-t3:.2f}s)")
print(f"⏱️ TOTAL: {t4-t1:.2f}s")

ipd.Audio(out)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

✅ Audio: 49152 samples at 16000Hz
📝 Transcript (0.43s):  Explain Newton's first law of motion.
🤖 Response (0.26s): Newton's first law of motion states that an object at rest will remain at rest, and an object in motion will continue to move with a constant velocity, unless acted upon by an external force. This means that an object will only change its velocity if a force is applied to it. It's also known as the law of inertia.
🔊 TTS (2.34s)
⏱️ TOTAL: 3.03s


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [11]:
# Create test audio
gTTS("Explain Newton's first law of motion", lang='en').save("/tmp/test_input.mp3")
os.system("ffmpeg -y -i /tmp/test_input.mp3 -ar 16000 /tmp/test_input.wav")

# Load it
audio_data, sr = sf.read("/tmp/test_input.wav")
print(f"✅ Audio: {len(audio_data)} samples at {sr}Hz")

# ASR
t1 = time.time()
segments, _ = asr_model.transcribe("/tmp/test_input.wav", beam_size=5)
transcript = " ".join([s.text for s in segments])
t2 = time.time()
print(f"📝 Transcript ({t2-t1:.2f}s): {transcript}")

# LLM
response_text = get_llm_response(transcript)
t3 = time.time()
print(f"🤖 Response ({t3-t2:.2f}s): {response_text}")

# TTS
out = text_to_speech(response_text)
t4 = time.time()
print(f"🔊 TTS ({t4-t3:.2f}s)")
print(f"⏱️ TOTAL: {t4-t1:.2f}s")

ipd.Audio(out)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

✅ Audio: 49152 samples at 16000Hz
📝 Transcript (0.72s):  Explain Newton's first law of motion.
🤖 Response (0.32s): Newton's first law of motion states that an object remains at rest or in uniform motion unless acted upon by an external force. This means that an object will continue to move in a straight line if no force is applied to it. Imagine a car rolling on a flat road - it will keep moving unless friction, a force, slows it down.
🔊 TTS (4.27s)
⏱️ TOTAL: 5.30s


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [13]:
import gradio as gr
import traceback

# Your voice assistant function
def voice_assistant(audio):
    try:
        # Example placeholder logic
        question_text = "What is photosynthesis?"
        answer_text = (
            "Photosynthesis is the process by which plants make food using sunlight."
        )

        # Replace this with generated audio path or numpy audio output
        response_audio = None

        return response_audio, question_text, answer_text

    except Exception as e:
        traceback.print_exc()
        return None, f"Error: {str(e)}", "Please try again"


# Create Gradio interface
app = gr.Interface(
    fn=voice_assistant,

    inputs=gr.Audio(
        sources=["upload"],
        type="numpy",
        label="📁 Upload WAV Audio File"
    ),

    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],

    title="🎓 AI Study Tutor — Voice Assistant",

    description="""
Upload a WAV audio file with your question
and get an instant AI tutor response!
""",

    examples=[],

    # Updated parameter for Gradio 4+
    flagging_mode="never"
)

# Launch app
app.launch(share=True)

* Running on local URL:  http://0.0.0.0:6007
* Running on public URL: https://b0f6aee484b37c82d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
try:
    app.close()
except:
    pass

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Record or Upload your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Record your question and get an instant voice response from your AI tutor!",
    flagging_mode="never"  # ← fixed parameter name
)

app.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True  # keep share=True for public URL
)

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://116f9a1fc70c68fef4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
try:
    app.close()
except:
    pass

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Record or Upload your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Record your question and get an instant voice response from your AI tutor!",
    flagging_mode="never"
)

app.launch(
    server_name="0.0.0.0",
    server_port=6006,  # ← JarvisLabs uses port 6006
    share=False
)

Closing server running on port: 7860


OSError: Cannot find empty port in range: 6006-6006. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

In [22]:
import os
import signal

# Force kill all ports
os.system("fuser -k 6006/tcp 2>/dev/null")
os.system("fuser -k 7860/tcp 2>/dev/null") 
os.system("fuser -k 8080/tcp 2>/dev/null")

# Kill any existing gradio processes
os.system("pkill -f gradio 2>/dev/null")
os.system("pkill -f uvicorn 2>/dev/null")

import time
time.sleep(3)
print("✅ All ports cleared")

# Check what's running
os.system("netstat -tlnp 2>/dev/null | grep -E '6006|7860|8080'")

✅ All ports cleared


256

In [23]:
try:
    app.close()
except:
    pass

import time
time.sleep(2)

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Record or Upload your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Record your question and get an instant voice response!",
    flagging_mode="never"
)

# Just use share=True - simpler and works
app.launch(share=True)

* Running on local URL:  http://0.0.0.0:6007
* Running on public URL: https://5e0bf8b9ed956aaf35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:
import uuid

def voice_assistant(audio):
    try:
        if audio is None:
            return None, "No audio received", "Please record something"
        
        sample_rate, audio_data = audio
        
        # Convert stereo to mono
        if len(audio_data.shape) > 1:
            audio_data = audio_data.mean(axis=1)
        
        # Normalize
        audio_data = audio_data.astype(np.float32)
        if np.abs(audio_data).max() > 1.0:
            audio_data = audio_data / 32768.0
        
        # Save with UNIQUE filename each time ← key fix
        unique_id = str(uuid.uuid4())[:8]
        tmp_wav = f"/tmp/input_{unique_id}.wav"
        sf.write(tmp_wav, audio_data, sample_rate)
        print(f"✅ Saved: {tmp_wav} at {sample_rate}Hz")
        
        # ASR
        t1 = time.time()
        segments, _ = asr_model.transcribe(tmp_wav, beam_size=5)
        transcript = " ".join([s.text for s in segments])
        t2 = time.time()
        print(f"📝 ASR ({t2-t1:.2f}s): {transcript}")
        
        if not transcript.strip():
            return None, "Could not hear anything", "Please speak clearly"
        
        # LLM
        response_text = get_llm_response(transcript)
        t3 = time.time()
        print(f"🤖 LLM ({t3-t2:.2f}s): {response_text}")
        
        # TTS with unique filename
        mp3_path = f"/tmp/response_{unique_id}.mp3"
        wav_path = f"/tmp/response_{unique_id}.wav"
        gTTS(text=response_text, lang='en').save(mp3_path)
        os.system(f"ffmpeg -y -i {mp3_path} -ar 22050 -ac 1 {wav_path}")
        t4 = time.time()
        print(f"⏱️ TOTAL: {t4-t1:.2f}s")
        
        return wav_path, f"You said: {transcript}", f"Tutor: {response_text}"
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"Error: {str(e)}", "Please try again"

# Relaunch
try:
    app.close()
except:
    pass

import time
time.sleep(2)

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Record or Upload your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Record your question and get an instant voice response from your AI tutor!",
    flagging_mode="never"
)

app.launch(share=True)

Closing server running on port: 6007
* Running on local URL:  http://0.0.0.0:6007
* Running on public URL: https://0f70d96d7d347be5e5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Saved: /tmp/input_42318ac2.wav at 44100Hz
📝 ASR (0.49s):  What is the capital of India?
🤖 LLM (0.22s): The capital of India is New Delhi. It serves as the country's political center and the seat of the Indian government. Located in the National Capital Territory of Delhi.


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

⏱️ TOTAL: 2.42s
✅ Saved: /tmp/input_56cd3652.wav at 44100Hz
📝 ASR (0.45s):  What is machine learning?
🤖 LLM (0.31s): Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It allows computers to improve their performance on a task over time, based on the data they receive. Think of it like a student improving their math skills through practice.


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

⏱️ TOTAL: 3.51s


In [25]:
%%writefile voice_assistant.py

import os
import time
import uuid
import numpy as np
import soundfile as sf
from gtts import gTTS
from faster_whisper import WhisperModel
from groq import Groq
import gradio as gr

# Load ASR
asr_model = WhisperModel("medium", device="cuda", compute_type="float16")

# Setup LLM
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def get_llm_response(text):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are a helpful study tutor assistant. 
            Help students understand concepts clearly and concisely.
            Keep responses under 3 sentences for voice output."""},
            {"role": "user", "content": text}
        ]
    )
    return response.choices[0].message.content

def text_to_speech(text, unique_id):
    mp3_path = f"/tmp/response_{unique_id}.mp3"
    wav_path = f"/tmp/response_{unique_id}.wav"
    gTTS(text=text, lang='en').save(mp3_path)
    os.system(f"ffmpeg -y -i {mp3_path} -ar 22050 -ac 1 {wav_path}")
    return wav_path

def voice_assistant(audio):
    try:
        if audio is None:
            return None, "No audio received", "Please record something"
        
        sample_rate, audio_data = audio
        if len(audio_data.shape) > 1:
            audio_data = audio_data.mean(axis=1)
        audio_data = audio_data.astype(np.float32)
        if np.abs(audio_data).max() > 1.0:
            audio_data = audio_data / 32768.0
        
        unique_id = str(uuid.uuid4())[:8]
        tmp_wav = f"/tmp/input_{unique_id}.wav"
        sf.write(tmp_wav, audio_data, sample_rate)
        
        t1 = time.time()
        segments, _ = asr_model.transcribe(tmp_wav, beam_size=5)
        transcript = " ".join([s.text for s in segments])
        t2 = time.time()
        
        if not transcript.strip():
            return None, "Could not hear anything", "Please speak clearly"
        
        response_text = get_llm_response(transcript)
        t3 = time.time()
        
        output_file = text_to_speech(response_text, unique_id)
        t4 = time.time()
        
        print(f"ASR: {t2-t1:.2f}s | LLM: {t3-t2:.2f}s | TTS: {t4-t3:.2f}s | Total: {t4-t1:.2f}s")
        
        return output_file, f"You said: {transcript}", f"Tutor: {response_text}"
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"Error: {str(e)}", "Please try again"

app = gr.Interface(
    fn=voice_assistant,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="numpy",
        label="🎤 Record or Upload your question"
    ),
    outputs=[
        gr.Audio(label="🔊 Tutor Response"),
        gr.Textbox(label="📝 Your Question"),
        gr.Textbox(label="🤖 Tutor Answer")
    ],
    title="🎓 AI Study Tutor — Voice Assistant",
    description="Record your question and get an instant voice response from your AI tutor!",
    flagging_mode="never"
)

if __name__ == "__main__":
    app.launch(share=True)

Writing voice_assistant.py


In [26]:
%%writefile requirements.txt
faster-whisper
gradio
groq
gTTS
soundfile
numpy
torch
torchaudio

Writing requirements.txt


In [30]:


# Create requirements.txt
cat > requirements.txt << 'EOF'
faster-whisper
gradio
groq
gTTS
soundfile
numpy
torch
torchaudio
EOF

echo "✅ requirements.txt created"
ls -la

SyntaxError: invalid syntax (2178997310.py, line 13)